# Lab 2 — Un agent à règles (correction)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/data_science_practice/blob/main/website/public/modules/python-medecine/notebooks/pym-s8-lab-2-correction.ipynb)

*Python pour la médecine — Séance 8, projet Puissance 4.*


**Objectifs.** Passer de l'observation à une grille simple ; écrire
des fonctions courtes et testables (`coups_valides`, `ligne_libre`,
`simuler_coup`, `est_gagnant`, `coup_gagnant`) ; les assembler en un
agent à règles — gagner, sinon bloquer, sinon ne pas offrir la
victoire, de préférence au centre — qui bat l'agent aléatoire presque
à tous les coups.

**Comment travailler.** Exécutez les cellules dans l'ordre. Une cellule
marquée `# À vous` contient un squelette (`...`) à compléter ; la
cellule suivante vérifie votre réponse et affiche « ✅ Exercice N
réussi ». Les cellules « Lire du code » demandent d'écrire votre
prédiction **avant** d'exécuter.

## 0. Installation

In [ ]:
import subprocess
import sys

try:
    import pettingzoo
except ImportError:  # Colab : PettingZoo n'est pas préinstallé
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pettingzoo[classic]"], check=True)

import random
import time

import matplotlib.pyplot as plt
import numpy as np
from pettingzoo.classic import connect_four_v3

random.seed(0)
print("PettingZoo prêt")

### Le code des labs précédents

La cellule suivante reprend la correction des lab 1 : exécutez-la
sans la modifier (ou remplacez-la par votre propre code).

In [ ]:
def afficher_plateau(observation):
    """Le plateau en texte : X = mes pions, O = l'adversaire, . = vide."""
    obs = np.asarray(observation)
    lignes = []
    for ligne in range(6):
        cases = []
        for col in range(7):
            if obs[ligne, col, 0] == 1:
                cases.append("X")
            elif obs[ligne, col, 1] == 1:
                cases.append("O")
            else:
                cases.append(".")
        lignes.append(" ".join(cases))
    return "\n".join(lignes)

def colonnes_libres(action_mask):
    """Les indices des colonnes autorisées (masque à 1)."""
    return [col for col, autorise in enumerate(action_mask) if autorise]

class AgentAleatoire:
    """Joue une colonne autorisée au hasard (le contrat ML-Arena)."""

    def setup(self, observation_space, action_space):
        return True

    def reset(self, env_player_name, episode_index):
        self.env_player_name = env_player_name
        self.episode_index = episode_index
        return True

    def choose_action(self, observation, reward=0.0, terminated=False,
                      truncated=False, info=None, action_mask=None):
        if terminated or truncated:
            return None
        return random.choice(colonnes_libres(action_mask))

class AgentPondere(AgentAleatoire):
    """Au hasard, mais les colonnes du centre sont plus probables."""

    POIDS = [1, 2, 3, 4, 3, 2, 1]

    def choose_action(self, observation, reward=0.0, terminated=False,
                      truncated=False, info=None, action_mask=None):
        if terminated or truncated:
            return None
        colonnes = colonnes_libres(action_mask)
        poids = [self.POIDS[col] for col in colonnes]
        return random.choices(colonnes, weights=poids)[0]

def jouer_partie(agent_a, agent_b, a_commence=True, index=0):
    """Une partie appelée comme sur ML-Arena : (résultat de agent_a, nb de coups).

    Le résultat vaut +1 (agent_a gagne), 0 (nulle) ou -1 (agent_a perd).
    """
    env = connect_four_v3.env()
    env.reset()
    if a_commence:
        places = {"player_0": agent_a, "player_1": agent_b}
    else:
        places = {"player_0": agent_b, "player_1": agent_a}
    for nom, agent in places.items():
        agent.reset(nom, index)
    gains = {nom: 0.0 for nom in env.possible_agents}
    nb_coups = 0
    for nom in env.agent_iter():
        observation, reward, terminated, truncated, info = env.last()
        if terminated or truncated:
            env.step(None)  # un joueur dont la partie est finie sort ainsi
            continue
        action = places[nom].choose_action(
            observation["observation"], float(reward), False, False, info,
            action_mask=observation["action_mask"])
        env.step(action)
        nb_coups += 1
        for joueur, gain in env.rewards.items():
            gains[joueur] += gain
    env.close()
    nom_a = "player_0" if a_commence else "player_1"
    return gains[nom_a], nb_coups

def jouer_match(agent_a, agent_b, n_parties=100):
    """n parties, agent_a commence une partie sur deux. Bilan vu par agent_a."""
    env = connect_four_v3.env()
    for agent in (agent_a, agent_b):  # une fois par match, comme sur ML-Arena
        agent.setup(env.observation_space("player_0"),
                    env.action_space("player_0"))
    bilan = {"victoires": 0, "nuls": 0, "defaites": 0, "coups": []}
    for i in range(n_parties):
        resultat, nb_coups = jouer_partie(agent_a, agent_b,
                                          a_commence=(i % 2 == 0), index=i)
        if resultat > 0:
            bilan["victoires"] += 1
        elif resultat < 0:
            bilan["defaites"] += 1
        else:
            bilan["nuls"] += 1
        bilan["coups"].append(nb_coups)
    return bilan

---

## 1. Une grille plus simple à manipuler

Deux canaux, c'est pratique pour un réseau de neurones, moins pour
raisonner. On travaille sur une grille 6 × 7 d'entiers :
`0` = vide, `1` = moi, `2` = l'adversaire.

In [ ]:
LIGNES, COLONNES = 6, 7
VIDE, MOI, ADV = 0, 1, 2

### Exercice 1 — `vers_grille`

Écrivez `vers_grille(observation)` en **une** opération NumPy : le
canal 0 vaut 1 là où j'ai un pion, le canal 1 là où l'adversaire en a
un. Renvoyez un tableau d'entiers (`.astype(int)`).

In [ ]:
def vers_grille(observation):
    """(6, 7, 2) -> grille 6 x 7 : 0 = vide, 1 = moi, 2 = l'adversaire."""
    obs = np.asarray(observation)
    return (obs[:, :, 0] + 2 * obs[:, :, 1]).astype(int)

In [ ]:
essai = np.zeros((6, 7, 2), dtype=np.int8)
essai[5, 3, 0] = 1
essai[4, 3, 1] = 1
g = vers_grille(essai)
assert g.shape == (6, 7)
assert g[5, 3] == MOI and g[4, 3] == ADV and g.sum() == 3
assert (vers_grille(essai.tolist()) == g).all()
print("✅ Exercice 1 réussi")

Trois outils pour la suite (donnés) : construire une grille à partir
d'un dessin en texte, et revenir à l'observation et au masque que
reçoit un agent — pour le tester sur des positions choisies.

In [ ]:
def grille_depuis_texte(texte):
    """Une grille depuis 6 lignes de texte : X = moi, O = l'adversaire."""
    symboles = {".": VIDE, "X": MOI, "O": ADV}
    lignes = [l.split() for l in texte.strip().splitlines()]
    return np.array([[symboles[s] for s in l] for l in lignes], dtype=int)


def observation_depuis_grille(grille):
    """L'inverse de vers_grille : l'observation (6, 7, 2) que reçoit l'agent."""
    return np.stack([grille == MOI, grille == ADV], axis=-1).astype(np.int8)


def masque_depuis_grille(grille):
    """Le masque des colonnes non pleines, comme action_mask."""
    return (grille[0] == VIDE).astype(np.int8)


g = grille_depuis_texte('''
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . O . . .
. . O X X . .
''')
print(g)
print(masque_depuis_grille(g))

---

## 2. Jouer un coup sur la grille

### Exercice 2 — `coups_valides`

Une colonne est jouable si sa case du **haut** (ligne 0) est vide.

In [ ]:
def coups_valides(grille):
    """Les colonnes dont la case du haut est vide."""
    return [col for col in range(COLONNES) if grille[0, col] == VIDE]

In [ ]:
assert coups_valides(np.zeros((6, 7), dtype=int)) == [0, 1, 2, 3, 4, 5, 6]
pleine_0_et_6 = np.zeros((6, 7), dtype=int)
pleine_0_et_6[:, 0] = [MOI, ADV] * 3
pleine_0_et_6[:, 6] = [ADV, MOI] * 3
assert coups_valides(pleine_0_et_6) == [1, 2, 3, 4, 5]
print("✅ Exercice 2 réussi")

### Exercice 3 — `ligne_libre`

La ligne où tombe un pion lâché dans `col` : la case vide la plus
**basse**. Parcourez les lignes de 5 à 0 avec
`range(LIGNES - 1, -1, -1)` ; renvoyez `None` si la colonne est pleine.

In [ ]:
def ligne_libre(grille, col):
    """La ligne où tombe un pion lâché dans col, ou None si elle est pleine."""
    for ligne in range(LIGNES - 1, -1, -1):  # du bas (5) vers le haut (0)
        if grille[ligne, col] == VIDE:
            return ligne
    return None

In [ ]:
vide = np.zeros((6, 7), dtype=int)
assert ligne_libre(vide, 3) == 5, "sur une grille vide, le pion tombe en bas"
assert ligne_libre(g, 3) == 3 and ligne_libre(g, 2) == 4 and ligne_libre(g, 0) == 5
assert ligne_libre(pleine_0_et_6, 0) is None
print("✅ Exercice 3 réussi")

### Lire du code 1

Pour essayer un coup sans abîmer la vraie grille, il faut une copie.
Que va afficher ce code ?

In [ ]:
a = np.zeros((6, 7), dtype=int)
b = a
b[5, 3] = MOI
print(a[5, 3])
c = a.copy()
c[5, 4] = ADV
print(a[5, 4])

**Réponse.** `1` puis `0`. `b = a` ne copie rien : `a` et `b` sont deux
noms pour **le même** tableau, modifier `b` modifie `a`. `a.copy()`
crée un nouveau tableau, indépendant.

### Exercice 4 — `simuler_coup`

Renvoyez une **copie** de la grille où `joueur` a joué dans `col` ;
la grille d'origine ne doit pas changer.

In [ ]:
def simuler_coup(grille, col, joueur):
    """Une COPIE de la grille où joueur a joué dans col."""
    nouvelle = grille.copy()
    nouvelle[ligne_libre(grille, col), col] = joueur
    return nouvelle

In [ ]:
avant = g.copy()
apres = simuler_coup(g, 3, MOI)
assert apres[3, 3] == MOI, "le pion se pose sur le O de la colonne 3"
assert (g == avant).all(), "la grille d'origine ne doit pas changer"
assert (apres != g).sum() == 1
print("✅ Exercice 4 réussi")

---

## 3. Détecter une victoire

![Les quatre directions](https://raw.githubusercontent.com/racousin/data_science_practice/main/courseware/content/python-medecine/assets/pym-s8-projet-puissance-4/directions-victoire.png)

Plutôt que de chercher autour de chaque pion, on énumère une fois pour
toutes les **fenêtres** : les suites de 4 cases alignées. Un joueur a
gagné si l'une d'elles ne contient que ses pions.

### Exercice 5 — Les 69 fenêtres

Pour chaque case de départ `(ligne, col)` et chaque direction
`(dl, dc)` parmi `(0, 1)` (horizontale), `(1, 0)` (verticale), `(1, 1)`
(diagonale ↘) et `(-1, 1)` (diagonale ↗), la fenêtre est
`[(ligne + k*dl, col + k*dc) for k in range(4)]` ; gardez-la si ses 4
cases sont dans la grille. Combien en attendez-vous ?

In [ ]:
def construire_fenetres():
    """Toutes les suites de 4 cases alignées sur la grille 6 x 7."""
    fenetres = []
    for ligne in range(LIGNES):
        for col in range(COLONNES):
            for dl, dc in [(0, 1), (1, 0), (1, 1), (-1, 1)]:
                cases = [(ligne + k * dl, col + k * dc) for k in range(4)]
                if all(0 <= l < LIGNES and 0 <= c < COLONNES for l, c in cases):
                    fenetres.append(cases)
    return fenetres


FENETRES = construire_fenetres()

print(len(FENETRES), "fenêtres")

In [ ]:
assert len(FENETRES) == 69, "24 horizontales + 21 verticales + 24 diagonales"
assert [(5, 0), (5, 1), (5, 2), (5, 3)] in FENETRES
assert [(5, 0), (4, 1), (3, 2), (2, 3)] in FENETRES, "diagonale ↗"
assert all(0 <= l < 6 and 0 <= c < 7 for f in FENETRES for l, c in f)
print("✅ Exercice 5 réussi")

### Exercice 6 — `est_gagnant`

Vrai si **toutes** les cases d'une fenêtre au moins valent `joueur`
(`any` et `all` font cela en une ligne, une boucle convient aussi).

In [ ]:
def est_gagnant(grille, joueur):
    """Vrai si joueur a quatre pions alignés quelque part."""
    for fenetre in FENETRES:
        if all(grille[l, c] == joueur for l, c in fenetre):
            return True
    return False

In [ ]:
horizontale = grille_depuis_texte('''
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. O O O . . .
. X X X X . .
''')
verticale = grille_depuis_texte('''
. . . . . . .
. . . . . . .
. . . . . O .
. . . . . O .
. X X . . O .
. X X . X O .
''')
diagonale = grille_depuis_texte('''
. . . . . . .
. . . . . . .
. . . X . . .
. . X O . . .
. X O O . . .
X O O X . . .
''')
antidiagonale = grille_depuis_texte('''
. . . . . . .
. . . . . . .
O . . . . . .
X O . . . . .
X X O . . . .
X O X O . . .
''')
assert est_gagnant(horizontale, MOI) and not est_gagnant(horizontale, ADV)
assert est_gagnant(verticale, ADV) and not est_gagnant(verticale, MOI)
assert est_gagnant(diagonale, MOI), "diagonale ↗"
assert est_gagnant(antidiagonale, ADV), "diagonale ↘"
assert not est_gagnant(g, MOI) and not est_gagnant(g, ADV)
print("✅ Exercice 6 réussi")

### Trouvez l'erreur

Cette fonction doit renvoyer une colonne qui fait gagner `joueur`
**tout de suite**, ou `None`. Sur la grille ci-dessous (deux X en bas),
aucun coup ne gagne. Que renvoie-t-elle, et qu'est-il arrivé à la
grille ?

In [ ]:
def coup_gagnant_bug(grille, joueur):
    for col in coups_valides(grille):
        grille[ligne_libre(grille, col), col] = joueur
        if est_gagnant(grille, joueur):
            return col
    return None

deux_x = grille_depuis_texte('''
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
O O . . . . .
X X . . . . .
''')
print(coup_gagnant_bug(deux_x, MOI))
print(deux_x)

**Réponse.** Elle renvoie `3` : les essais précédents (colonnes 0, 1
et 2) sont restés sur la grille, si bien qu'au quatrième la ligne du
bas contient quatre X. La fonction **modifie** la grille qu'on lui
passe au lieu de travailler sur une copie : il faut `simuler_coup`.

### Exercice 7 — `coup_gagnant`

La version correcte, avec `simuler_coup`.

In [ ]:
def coup_gagnant(grille, joueur):
    """Une colonne qui fait gagner joueur tout de suite, sinon None."""
    for col in coups_valides(grille):
        if est_gagnant(simuler_coup(grille, col, joueur), joueur):
            return col
    return None

In [ ]:
deux_x = grille_depuis_texte('''
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
O O . . . . .
X X . . . . .
''')
copie = deux_x.copy()
assert coup_gagnant(deux_x, MOI) is None
assert (deux_x == copie).all(), "la grille ne doit pas changer"
assert coup_gagnant(simuler_coup(deux_x, 2, MOI), MOI) == 3
trois_o = grille_depuis_texte('''
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . O
. . . . X . O
. . . X X X O
''')
assert coup_gagnant(trois_o, ADV) == 6
assert coup_gagnant(trois_o, MOI) in (2, 6)
print("✅ Exercice 7 réussi")

---

## 4. L'agent à règles

Les règles, dans l'ordre de priorité :

1. si je peux gagner, je gagne ;
2. sinon, si l'adversaire peut gagner au prochain coup, je bloque ;
3. sinon, j'évite les coups **suicidaires** : ceux après lesquels
   l'adversaire gagne (typiquement, jouer juste sous la case qui lui
   manque) ;
4. parmi les coups restants, je tire au hasard en favorisant le centre.

### Exercice 8 — `coups_surs`

Gardez les coups de `coups` après lesquels l'adversaire n'a **pas** de
coup gagnant.

In [ ]:
def coups_surs(grille, coups):
    """Les coups après lesquels l'adversaire ne peut pas gagner tout de suite."""
    return [col for col in coups
            if coup_gagnant(simuler_coup(grille, col, MOI), ADV) is None]

In [ ]:
piege = grille_depuis_texte('''
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. O O O X . .
. X O X X O .
''')
# Les trois O de la ligne 4 n'attendent que la case (4, 0), encore en
# l'air : jouer en colonne 0 la leur offre.
assert 0 not in coups_surs(piege, coups_valides(piege))
assert 4 in coups_surs(piege, coups_valides(piege))
assert coups_surs(np.zeros((6, 7), dtype=int), list(range(7))) == list(range(7))
print("✅ Exercice 8 réussi")

### Exercice 9 — `AgentRegles`

Assemblez les quatre règles dans `choose_action`. La classe hérite
d'`AgentAleatoire` (donc de `setup` et `reset`). Commencez par
`grille = vers_grille(observation)` et
`coups = colonnes_libres(action_mask)`.

In [ ]:
class AgentRegles(AgentAleatoire):
    """Gagner, sinon bloquer, sinon un coup sûr de préférence au centre."""

    POIDS = [1, 2, 3, 4, 3, 2, 1]

    def choose_action(self, observation, reward=0.0, terminated=False,
                      truncated=False, info=None, action_mask=None):
        if terminated or truncated:
            return None
        grille = vers_grille(observation)
        coups = colonnes_libres(action_mask)
        for joueur in (MOI, ADV):  # 1. gagner, 2. bloquer
            col = coup_gagnant(grille, joueur)
            if col is not None:
                return col
        candidats = coups_surs(grille, coups) or coups  # 3. ne pas offrir
        poids = [self.POIDS[col] for col in candidats]  # 4. le centre
        return random.choices(candidats, weights=poids)[0]

In [ ]:
def coup_de(agent, grille):
    return agent.choose_action(observation_depuis_grille(grille),
                               action_mask=masque_depuis_grille(grille))

agent = AgentRegles()
gagner = grille_depuis_texte('''
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. O O . . . .
. X X X O . .
''')
assert coup_de(agent, gagner) == 0, "X X X : gagner en colonne 0"
bloquer = grille_depuis_texte('''
. . . . . . .
. . . . . . .
. . . . . . .
. . . O . . .
. X . O . . .
. X X O . . .
''')
assert coup_de(agent, bloquer) == 3, "bloquer la verticale des O"
assert all(coup_de(agent, piege) != 0 for _ in range(30)), "ne pas offrir (4, 0)"
bilan = jouer_match(AgentRegles(), AgentAleatoire(), 100)
print("règles contre aléatoire :", bilan["victoires"], "victoires sur 100")
assert bilan["victoires"] >= 90
print("✅ Exercice 9 réussi")

### Lire du code 2

Sur cette grille, je peux gagner en colonne 6 **et** l'adversaire peut
gagner en colonne 0. Que joue `AgentRegles` ? Pourquoi est-ce le bon
ordre ?

In [ ]:
les_deux = grille_depuis_texte('''
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . X
. . . . . . X
. O O O . . X
''')
print(coup_de(AgentRegles(), les_deux))

**Réponse.** `6`. La règle « gagner » passe avant « bloquer » : gagner
termine la partie, l'adversaire n'aura jamais l'occasion de jouer sa
colonne 0. Bloquer d'abord laisserait passer une victoire certaine.

---

## 5. Pour aller plus loin (facultatif)

### Exercice 10 — La double menace

Une **double menace** est un coup après lequel j'ai **deux** colonnes
gagnantes différentes : l'adversaire ne peut en bloquer qu'une.
Écrivez `cree_double_menace(grille, col, joueur)`. Puis ajoutez cette
règle à un `AgentReglesPlus(AgentRegles)` (entre « bloquer » et « coup
sûr ») et mesurez-le contre `AgentRegles`.

In [ ]:
def cree_double_menace(grille, col, joueur):
    """Vrai si, après joueur en col, joueur a au moins deux colonnes gagnantes."""
    apres = simuler_coup(grille, col, joueur)
    gagnantes = [c for c in coups_valides(apres)
                 if est_gagnant(simuler_coup(apres, c, joueur), joueur)]
    return len(gagnantes) >= 2


class AgentReglesPlus(AgentRegles):
    """AgentRegles + créer une double menace quand c'est possible."""

    def choose_action(self, observation, reward=0.0, terminated=False,
                      truncated=False, info=None, action_mask=None):
        if terminated or truncated:
            return None
        grille = vers_grille(observation)
        coups = colonnes_libres(action_mask)
        for joueur in (MOI, ADV):
            col = coup_gagnant(grille, joueur)
            if col is not None:
                return col
        surs = coups_surs(grille, coups)
        for col in surs:
            if cree_double_menace(grille, col, MOI):
                return col
        candidats = surs or coups
        poids = [self.POIDS[col] for col in candidats]
        return random.choices(candidats, weights=poids)[0]


bilan = jouer_match(AgentReglesPlus(), AgentRegles(), 60)
print("règles+ contre règles :", bilan["victoires"], "victoires,",
      bilan["defaites"], "défaites sur 60")

In [ ]:
ouverte = grille_depuis_texte('''
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. O O . . . .
. X X . . O .
''')
assert cree_double_menace(ouverte, 3, MOI), ". X X X . : deux cases gagnantes"
assert not cree_double_menace(ouverte, 0, MOI)
print("✅ Exercice 10 réussi")

---

**Bilan.** Des fonctions courtes, chacune vérifiée sur des grilles
choisies, puis assemblées : c'est ce qui rend l'agent fiable. Le lab 3
transforme ces vérifications en vrais tests (pytest) et organise un
tournoi.